In [ ]:
from pathlib import Path
import os
import random
import zipfile
import shutil
import time
import json
import csv
import math
import hashlib
from collections import defaultdict
from io import BytesIO
from PIL import Image, ImageOps


In [ ]:
SEED = 42
SOURCE_ZIP = Path("VinaFood21.zip")
LOCAL_OUT = Path("VinaFood21_resized")
LOCAL_ZIP = Path("VinaFood21_resized.zip")
ON_KAGGLE = Path("/kaggle/input").exists()
PREPARED_ROOT = (
    Path("/kaggle/working/VinaFood21") if ON_KAGGLE else LOCAL_OUT / "VinaFood21"
)


In [ ]:
def prepare_resized_dataset(
    zip_path=SOURCE_ZIP, output_dir=LOCAL_OUT, side=224, quality=88
):
    zip_path, output_dir = (Path(zip_path), Path(output_dir))
    if not zip_path.is_file():
        raise FileNotFoundError(
            f"Không thấy {zip_path.resolve()}; đặt VinaFood21.zip cạnh notebook."
        )
    if output_dir.exists():
        shutil.rmtree(output_dir)
    out_root = output_dir / "VinaFood21"
    groups = defaultdict(list)
    image_ext = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    with zipfile.ZipFile(zip_path) as zf:
        for name in zf.namelist():
            p = Path(name)
            parts = [x.lower() for x in p.parts]
            if p.suffix.lower() not in image_ext:
                continue
            split_i = next(
                (
                    i
                    for i, x in enumerate(parts)
                    if x
                    in {
                        "train",
                        "training",
                        "test",
                        "testing",
                        "val",
                        "valid",
                        "validation",
                    }
                ),
                None,
            )
            if split_i is None or split_i + 1 >= len(p.parts) - 1:
                continue
            split_name = parts[split_i]
            split = (
                "train"
                if split_name in {"train", "training"}
                else "test" if split_name in {"test", "testing"} else "val"
            )
            label = p.parts[split_i + 1]
            groups[split, label].append(name)
        if not any((k[0] == "train" for k in groups)):
            raise ValueError(
                "Không tìm thấy thư mục train/<class> trong ZIP. Kiểm tra cấu trúc archive."
            )
        rng = random.Random(SEED)
        train_files, val_files = ([], [])
        labels = sorted({label for split, label in groups if split == "train"})
        for label in labels:
            items = sorted(groups["train", label])
            rng.shuffle(items)
            n_val = max(1, round(0.2 * len(items))) if len(items) > 1 else 0
            val_files.extend(((name, label) for name in items[:n_val]))
            train_files.extend(((name, label) for name in items[n_val:]))
        jobs = [("train", n, c) for n, c in train_files] + [
            ("val", n, c) for n, c in val_files
        ]
        jobs += [
            ("test", n, c)
            for (split, label), names in groups.items()
            if split == "test"
            for n in names
            for c in [label]
        ]
        if any((k[0] == "val" for k in groups)):
            jobs = [
                ("train", n, c)
                for (s, c), names in groups.items()
                if s == "train"
                for n in names
            ]
            jobs += [
                ("val", n, c)
                for (s, c), names in groups.items()
                if s == "val"
                for n in names
            ]
            jobs += [
                ("test", n, c)
                for (s, c), names in groups.items()
                if s == "test"
                for n in names
            ]
        errors = []
        for split, name, label in jobs:
            try:
                with zf.open(name) as f, Image.open(f) as image:
                    image = ImageOps.exif_transpose(image)
                    if image.mode in {"RGBA", "LA"} or "transparency" in image.info:
                        rgba = image.convert("RGBA")
                        rgb = Image.new("RGB", rgba.size, (255, 255, 255))
                        rgb.paste(rgba, mask=rgba.getchannel("A"))
                        image = rgb
                    else:
                        image = image.convert("RGB")
                    image = image.resize((side, side), Image.Resampling.LANCZOS)
                    target = (
                        out_root
                        / split
                        / label
                        / f"{Path(name).stem}_{hashlib.sha1(name.encode()).hexdigest()[:8]}.jpg"
                    )
                    target.parent.mkdir(parents=True, exist_ok=True)
                    image.save(target, format="JPEG", quality=quality, optimize=True)
            except Exception as exc:
                errors.append((name, type(exc).__name__, str(exc)))
        if errors:
            raise RuntimeError(f"{len(errors)} ảnh lỗi; ví dụ: {errors[:3]}")
    counts = {
        s: sum((1 for p in (out_root / s).glob("*/*") if p.is_file()))
        for s in ("train", "val", "test")
    }
    if len(labels) != 21 or any((counts[s] == 0 for s in ("train", "val", "test"))):
        raise AssertionError(
            f"Cần 21 lớp và đủ train/val/test; labels={len(labels)}, counts={counts}"
        )
    for split in ("train", "val", "test"):
        split_root = out_root / split
        present = {p.name for p in split_root.iterdir() if p.is_dir()}
        if present != set(labels) or any(
            (not any((split_root / c).glob("*.jpg")) for c in labels)
        ):
            raise AssertionError(f"Thiếu lớp/ảnh trong {split}: {present}")
    sample = next((out_root / "train").glob("*/*.jpg"))
    with Image.open(sample) as im:
        if im.size != (side, side) or im.mode != "RGB":
            raise AssertionError(f"Sai kích thước/mode: {im.size}, {im.mode}")
    if output_dir.exists():
        output_dir.rmdir() if not any(output_dir.iterdir()) else None
    return (out_root, counts)


In [ ]:
if not ON_KAGGLE:
    DATA_ROOT, split_counts = prepare_resized_dataset()
    print("Đã resize:", DATA_ROOT.resolve(), split_counts)
    if LOCAL_ZIP.exists():
        LOCAL_ZIP.unlink()
    with zipfile.ZipFile(LOCAL_ZIP, "w", compression=zipfile.ZIP_STORED) as zf:
        for path in DATA_ROOT.rglob("*"):
            if path.is_file():
                zf.write(path, path.relative_to(LOCAL_OUT))
    print(
        f"Dataset upload: {LOCAL_ZIP.resolve()} ({LOCAL_ZIP.stat().st_size / 1024 ** 2:.1f} MiB)"
    )
else:
    KAGGLE_DATASET_ROOT = Path(
        "/kaggle/input/datasets/bowboochua9/ds201-lab2-vinafood21-224"
    )
    DATA_ROOT = KAGGLE_DATASET_ROOT / "VinaFood21"
    required_splits = ("train", "val", "test")
    if not DATA_ROOT.is_dir() or any(
        (not (DATA_ROOT / split).is_dir() for split in required_splits)
    ):
        raise FileNotFoundError(
            f"Kaggle dataset mount không đúng cấu trúc; cần {DATA_ROOT}/{{train,val,test}}. Input roots: {list(Path('/kaggle/input').iterdir())}"
        )
    print("Dataset Kaggle đã xác nhận:", DATA_ROOT)


In [ ]:
import importlib.metadata as package_meta
import subprocess
import sys

if ON_KAGGLE:
    try:
        transformers_version = package_meta.version("transformers")
        hub_version = package_meta.version("huggingface-hub")
    except package_meta.PackageNotFoundError:
        subprocess.check_call(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "-q",
                "transformers==4.48.1",
                "huggingface-hub<1.0",
            ]
        )
    else:
        if (
            int(transformers_version.split(".")[0]) < 5
            and int(hub_version.split(".")[0]) >= 1
        ):
            subprocess.check_call(
                [sys.executable, "-m", "pip", "install", "-q", "huggingface-hub<1.0"]
            )

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
)
import matplotlib.pyplot as plt


In [ ]:
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if ON_KAGGLE:
    if DEVICE.type != "cuda":
        raise RuntimeError(
            "Kaggle notebook has no GPU; refusing to start the full run."
        )
    GPU_NAME = torch.cuda.get_device_name(0)
    if "T4" not in GPU_NAME:
        raise RuntimeError(f"Expected Tesla T4, got {GPU_NAME}")
    print(
        "GPU:",
        GPU_NAME,
        "VRAM GiB:",
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1),
    )

CLASSES = sorted((p.name for p in (DATA_ROOT / "train").iterdir() if p.is_dir()))
assert len(CLASSES) == 21, f"Mong đợi 21 lớp, thấy {len(CLASSES)}"
MEAN, STD = ((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
EPOCHS = 10
RUN_EXPERIMENTS = ON_KAGGLE
OUT_DIR = (
    Path("/kaggle/working/ds201_lab2") if ON_KAGGLE else Path("ds201_lab2_outputs")
)
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(
    "device:",
    DEVICE,
    "classes:",
    len(CLASSES),
    "epochs:",
    EPOCHS,
    "run:",
    RUN_EXPERIMENTS,
)


In [ ]:
def make_loaders(size, augment=True, batch_size=32):
    norm = transforms.Normalize(MEAN, STD)
    train_tf = (
        transforms.Compose(
            [
                transforms.RandomResizedCrop(size, scale=(0.7, 1.0)),
                transforms.RandomHorizontalFlip(),
                transforms.RandomRotation(15),
                transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
                transforms.ToTensor(),
                norm,
            ]
        )
        if augment
        else transforms.Compose(
            [transforms.Resize((size, size)), transforms.ToTensor(), norm]
        )
    )
    eval_tf = transforms.Compose(
        [transforms.Resize((size, size)), transforms.ToTensor(), norm]
    )
    tr = datasets.ImageFolder(DATA_ROOT / "train", transform=train_tf)
    va = datasets.ImageFolder(DATA_ROOT / "val", transform=eval_tf)
    te = datasets.ImageFolder(DATA_ROOT / "test", transform=eval_tf)
    assert tr.classes == va.classes == te.classes == CLASSES
    kw = dict(num_workers=2, pin_memory=torch.cuda.is_available())
    return (
        DataLoader(tr, batch_size=batch_size, shuffle=True, **kw),
        DataLoader(va, batch_size=batch_size, shuffle=False, **kw),
        DataLoader(te, batch_size=batch_size, shuffle=False, **kw),
    )


In [ ]:
class LeNet5(nn.Module):

    def __init__(self, pool="avg", num_classes=21):
        super().__init__()
        Pool = nn.AvgPool2d if pool == "avg" else nn.MaxPool2d
        self.c1 = nn.Conv2d(3, 6, 5)
        self.s2 = Pool(2, 2)
        self.c3 = nn.Conv2d(6, 16, 5)
        self.s4 = Pool(2, 2)
        self.f5 = nn.Linear(16 * 13 * 13, 120)
        self.f6 = nn.Linear(120, 84)
        self.out = nn.Linear(84, num_classes)

    def forward(self, x):
        x = self.s2(F.relu(self.c1(x)))
        x = self.s4(F.relu(self.c3(x)))
        return self.out(F.relu(self.f6(F.relu(self.f5(x.flatten(1))))))


In [ ]:
class InceptionBlock(nn.Module):

    def __init__(self, c, f1, f3r, f3, f5r, f5, fp):
        super().__init__()
        self.b1 = nn.Sequential(nn.Conv2d(c, f1, 1), nn.ReLU())
        self.b2 = nn.Sequential(
            nn.Conv2d(c, f3r, 1), nn.ReLU(), nn.Conv2d(f3r, f3, 3, padding=1), nn.ReLU()
        )
        self.b3 = nn.Sequential(
            nn.Conv2d(c, f5r, 1), nn.ReLU(), nn.Conv2d(f5r, f5, 5, padding=2), nn.ReLU()
        )
        self.b4 = nn.Sequential(
            nn.MaxPool2d(3, stride=1, padding=1), nn.Conv2d(c, fp, 1), nn.ReLU()
        )
        self.out_channels = f1 + f3 + f5 + fp

    def forward(self, x):
        return torch.cat([self.b1(x), self.b2(x), self.b3(x), self.b4(x)], dim=1)


class GoogleNetSmall(nn.Module):

    def __init__(self, num_classes=21):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2)
        )
        self.blocks = nn.ModuleList(
            [
                InceptionBlock(32, 16, 16, 24, 4, 8, 8),
                InceptionBlock(56, 32, 32, 48, 8, 16, 16),
                InceptionBlock(112, 64, 48, 96, 8, 32, 32),
            ]
        )
        self.pool = nn.MaxPool2d(2)
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(224, num_classes)

    def forward(self, x):
        x = self.stem(x)
        for i, block in enumerate(self.blocks):
            x = block(x)
            if i < len(self.blocks) - 1:
                x = self.pool(x)
        return self.fc(self.gap(x).flatten(1))


In [ ]:
class BasicBlock(nn.Module):

    def __init__(self, cin, cout, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(cin, cout, 3, stride, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(cout)
        self.conv2 = nn.Conv2d(cout, cout, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(cout)
        self.shortcut = (
            nn.Identity()
            if stride == 1 and cin == cout
            else nn.Sequential(
                nn.Conv2d(cin, cout, 1, stride, bias=False), nn.BatchNorm2d(cout)
            )
        )

    def forward(self, x):
        return F.relu(
            self.bn2(self.conv2(F.relu(self.bn1(self.conv1(x))))) + self.shortcut(x)
        )


class ResNet18(nn.Module):

    def __init__(self, num_classes=21):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, 64, 7, 2, 3, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(3, 2, 1),
        )
        self.layer1 = self.stage(64, 64, 1)
        self.layer2 = self.stage(64, 128, 2)
        self.layer3 = self.stage(128, 256, 2)
        self.layer4 = self.stage(256, 512, 2)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(512, num_classes)

    @staticmethod
    def stage(cin, cout, stride):
        return nn.Sequential(BasicBlock(cin, cout, stride), BasicBlock(cout, cout))

    def forward(self, x):
        x = self.stem(x)
        for layer in (self.layer1, self.layer2, self.layer3, self.layer4):
            x = layer(x)
        return self.fc(self.pool(x).flatten(1))


In [ ]:
class PretrainedResnet(nn.Module):

    def __init__(self, freeze=False):
        super().__init__()
        from transformers import ResNetForImageClassification

        base = ResNetForImageClassification.from_pretrained("microsoft/resnet-50")
        self.backbone = base.resnet
        self.classifier = nn.Linear(2048, 21)
        self.freeze_backbone = freeze
        if freeze:
            for p in self.backbone.parameters():
                p.requires_grad = False

    def forward(self, x):
        feat = self.backbone(x).pooler_output.squeeze(-1).squeeze(-1)
        return self.classifier(feat)


def make_resnet50(freeze=False):
    return PretrainedResnet(freeze)


def n_params(model, trainable=False):
    return sum(
        (p.numel() for p in model.parameters() if p.requires_grad or not trainable)
    )


In [ ]:
@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    loss_sum = 0.0
    y_true = []
    y_pred = []
    for x, y in loader:
        x, y = (x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True))
        logits = model(x)
        loss_sum += criterion(logits, y).item() * len(y)
        y_true.extend(y.cpu().tolist())
        y_pred.extend(logits.argmax(1).cpu().tolist())
    p, r, f, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )
    return (
        loss_sum / len(loader.dataset),
        accuracy_score(y_true, y_pred),
        p,
        r,
        f,
        y_true,
        y_pred,
    )


def fit_one(name, model, size, augment, batch_size, lr, epochs=EPOCHS):
    train_dl, val_dl, test_dl = make_loaders(size, augment, batch_size)
    model = model.to(DEVICE)
    opt = torch.optim.Adam((p for p in model.parameters() if p.requires_grad), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    scaler = torch.amp.GradScaler("cuda", enabled=DEVICE.type == "cuda")
    best = -1.0
    history = []
    ckpt = OUT_DIR / f"{name}_best.pt"
    start = time.perf_counter()
    for ep in range(1, epochs + 1):
        model.train()
        if getattr(model, "freeze_backbone", False):
            model.backbone.eval()
        total = 0
        loss_total = 0.0
        for x, y in train_dl:
            x, y = (x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True))
            opt.zero_grad(set_to_none=True)
            with torch.autocast(
                device_type=DEVICE.type,
                dtype=torch.float16,
                enabled=DEVICE.type == "cuda",
            ):
                logits = model(x)
                loss = loss_fn(logits, y)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            loss_total += loss.item() * len(y)
            total += len(y)
        val_loss, val_acc, _, _, val_f1, _, _ = evaluate(model, val_dl, loss_fn)
        history.append(
            {
                "epoch": ep,
                "train_loss": loss_total / total,
                "val_loss": val_loss,
                "val_accuracy": val_acc,
                "val_macro_f1": val_f1,
            }
        )
        print(
            f"{name} ep {ep}/{epochs} train_loss={loss_total / total:.4f} val_loss={val_loss:.4f} val_acc={val_acc:.4f} val_macroF1={val_f1:.4f}"
        )
        if val_f1 > best:
            best = val_f1
            torch.save(model.state_dict(), ckpt)
    epoch_seconds = (time.perf_counter() - start) / epochs
    model.load_state_dict(torch.load(ckpt, map_location=DEVICE, weights_only=True))
    _, acc, prec, rec, f1, y_true, y_pred = evaluate(model, test_dl, loss_fn)
    report = classification_report(
        y_true, y_pred, target_names=CLASSES, output_dict=True, zero_division=0
    )
    cm = confusion_matrix(y_true, y_pred)
    pd.DataFrame(history).to_csv(OUT_DIR / f"{name}_history.csv", index=False)
    plt.figure()
    plt.plot(
        [h["epoch"] for h in history], [h["train_loss"] for h in history], label="train"
    )
    plt.plot(
        [h["epoch"] for h in history],
        [h["val_loss"] for h in history],
        label="validation",
    )
    plt.xlabel("epoch")
    plt.ylabel("loss")
    plt.legend()
    plt.tight_layout()
    plt.savefig(OUT_DIR / f"{name}_loss.png")
    plt.close()
    fig, ax = plt.subplots(figsize=(12, 10))
    im = ax.imshow(cm, cmap="Blues")
    fig.colorbar(im, ax=ax)
    ax.set(
        xticks=range(len(CLASSES)),
        yticks=range(len(CLASSES)),
        xticklabels=CLASSES,
        yticklabels=CLASSES,
        xlabel="Predicted",
        ylabel="True",
        title=name,
    )
    plt.setp(ax.get_xticklabels(), rotation=90, ha="right")
    fig.tight_layout()
    fig.savefig(OUT_DIR / f"{name}_confusion.png")
    plt.close(fig)
    pd.DataFrame(cm, index=CLASSES, columns=CLASSES).to_csv(
        OUT_DIR / f"{name}_confusion.csv"
    )
    pd.DataFrame(report).T.to_csv(OUT_DIR / f"{name}_classification_report.csv")
    result = {
        "experiment": name,
        "resize": size,
        "parameters": n_params(model),
        "trainable_parameters": n_params(model, True),
        "seconds_per_epoch": epoch_seconds,
        "accuracy": acc,
        "precision_macro": prec,
        "recall_macro": rec,
        "f1_macro": f1,
        "best_val_macro_f1": best,
    }
    print(
        classification_report(
            y_true, y_pred, target_names=CLASSES, digits=4, zero_division=0
        )
    )
    return result


In [ ]:
lenet_probe = LeNet5("avg")
x = torch.zeros(1, 3, 64, 64)
rows = []

for name, layer in [
    ("C1", lenet_probe.c1),
    ("S2", lenet_probe.s2),
    ("C3", lenet_probe.c3),
    ("S4", lenet_probe.s4),
]:
    x = layer(x) if name in {"S2", "S4"} else layer(x) if name in {"C1", "C3"} else x
    if name in {"C1", "C3"}:
        x = F.relu(x)
    rows.append((name, tuple(x.shape), n_params(layer)))

x = x.flatten(1)

for name, layer in [
    ("F5", lenet_probe.f5),
    ("F6", lenet_probe.f6),
    ("OUT", lenet_probe.out),
]:
    x = layer(x)
    rows.append((name, tuple(x.shape), n_params(layer)))

print("LeNet layer / shape / parameters:", rows, "TOTAL", n_params(lenet_probe))
g = GoogleNetSmall()
x = g.stem(torch.zeros(1, 3, 64, 64))

for i, b in enumerate(g.blocks, 1):
    x = b(x)
    print(
        f"Inception {i}: observed={x.shape[1]} calculated={b.out_channels} params={n_params(b)}"
    )
    assert x.shape[1] == b.out_channels
    if i < len(g.blocks):
        x = g.pool(x)


In [ ]:
if RUN_EXPERIMENTS:
    experiments = [
        ("lenet_avg", lambda: LeNet5("avg"), 64, True, 32, 0.001),
        ("lenet_max", lambda: LeNet5("max"), 64, True, 32, 0.001),
        ("googlenet_small", GoogleNetSmall, 64, True, 32, 0.001),
        ("resnet18_no_aug", ResNet18, 96, False, 32, 0.001),
        ("resnet18_aug", ResNet18, 96, True, 32, 0.001),
        (
            "resnet50_feature_extraction",
            lambda: make_resnet50(True),
            224,
            True,
            16,
            0.001,
        ),
        ("resnet50_finetune", lambda: make_resnet50(False), 224, True, 16, 0.0001),
    ]
    results = []
    for name, factory, size, aug, batch, lr in experiments:
        result = fit_one(name, factory(), size, aug, batch, lr)
        results.append(result)
        pd.DataFrame(results).to_csv(OUT_DIR / "summary.csv", index=False)
    display(pd.DataFrame(results))
    by = {r["experiment"]: r for r in results}
    print("\nMacro-F1 deltas:")
    print(
        "ResNet-18 augmentation:",
        round(by["resnet18_aug"]["f1_macro"] - by["resnet18_no_aug"]["f1_macro"], 4),
    )
    print(
        "ResNet-50 fine-tune vs feature extraction:",
        round(
            by["resnet50_finetune"]["f1_macro"]
            - by["resnet50_feature_extraction"]["f1_macro"],
            4,
        ),
    )
    print(
        "ResNet-50 fine-tune vs ResNet-18 augmented:",
        round(by["resnet50_finetune"]["f1_macro"] - by["resnet18_aug"]["f1_macro"], 4),
    )
    for name in ("resnet18_no_aug", "resnet18_aug"):
        h = pd.read_csv(OUT_DIR / f"{name}_history.csv")
        best_idx = int(h["val_loss"].idxmin())
        diverge = next(
            (
                int(h.iloc[i]["epoch"])
                for i in range(best_idx + 1, len(h))
                if h.iloc[i]["val_loss"] > h.iloc[i - 1]["val_loss"]
                and h.iloc[i]["train_loss"] < h.iloc[i - 1]["train_loss"]
            ),
            None,
        )
        print(
            f"{name}: validation loss first rises while train loss falls after its minimum at epoch {diverge}; None means not observed"
        )
else:
    checks = [(LeNet5(), 64), (GoogleNetSmall(), 64), (ResNet18(), 96)]
    for model, size in checks:
        with torch.no_grad():
            y = model(torch.randn(2, 3, size, size))
        assert y.shape == (2, 21)
        print(
            type(model).__name__,
            size,
            "→",
            tuple(y.shape),
            "params",
            f"{n_params(model):,}",
        )
    print(
        "Local mode: model smoke tests passed; set RUN_EXPERIMENTS=True only for a deliberate local run."
    )
